# COMPACT-MI-RFE — CICFLOW Colab Orchestrator

This notebook is a new orchestrator. It does not modify the reference notebook, the CTU-13 scripts, the Windows dataset, or completed Stage 4 runs.

Default mode is `dry_run`. MI/RFE fitting, resume, and Test evaluation are never triggered by Run All in the default configuration. Stage 4/5 remain `PROVISIONAL_RESEARCH` until the reference contract is reconciled.

## Scientific and data rules

- One dataset root only: `CICFLOWMETER_FEATURE_CSV`.
- Four configurations are supported.
- MI and RFE read Train only. Validation/Test are not opened in these stages.
- Stage 4 MI is bound to its manifest and identity digest. Stage 5A binds to the exact MI run.
- XGBoost-RFE is blocked until an estimator/ranking contract is verified.
- The notebook does not silently change StandardScaler, MI seeds, estimator parameters, or tie semantics.

In [ ]:
# Safe defaults: change explicitly before running a non-dry mode.
MODE = 'dry_run'  # dry_run | full | resume | verify
REFERENCE_RECONCILIATION_CONFIRMED = False
CONFIRM_PROVISIONAL_RESEARCH = False
INSTALL_DEPENDENCIES = False
MOUNT_DRIVE = False
BACKUP_SMALL_ARTIFACTS = False
GIT_COMMIT = 'NOT_CAPTURED'

PROJECT_ROOT = '/content/drive/MyDrive/COMPACT-MI-RFE'
DATASET_ROOT = '/content/drive/MyDrive/COMPACT-MI-RFE/CICFLOWMETER_FEATURE_CSV'
DRIVE_ROOT = '/content/drive/MyDrive'

CONFIGS = ['FW256_FS8', 'FW256_FS16', 'FW512_FS8', 'FW512_FS16']
BRANCHES = ['rf', 'svm']
RUN_IDS = {
    'FW256_FS8': 'RUN_MI_20261027_01',
    'FW256_FS16': 'RUN_MI_20261027_02',
    'FW512_FS8': 'RUN_MI_20261027_03',
    'FW512_FS16': 'RUN_MI_20261027_04',
}

if MODE not in {'dry_run', 'full', 'resume', 'verify'}:
    raise ValueError(f'Unsupported MODE: {MODE}')
if MODE in {'full', 'resume'} and not REFERENCE_RECONCILIATION_CONFIRMED:
    raise RuntimeError('Set REFERENCE_RECONCILIATION_CONFIRMED=True only after source reconciliation')
if MODE in {'full', 'resume'} and not CONFIRM_PROVISIONAL_RESEARCH:
    raise RuntimeError('Set CONFIRM_PROVISIONAL_RESEARCH=True for provisional full/resume runs')

## Optional Colab runtime setup

Run this cell only when intentionally mounting the Drive. It does not clone a repository or install packages by default. The source checkout and dataset package must be prepared by the user without copying unnecessary Test files.

In [ ]:
if MOUNT_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')

if INSTALL_DEPENDENCIES:
    # Pin versions only after reconciling them with the reference environment.
    get_ipython().run_line_magic('pip', 'install numpy==1.26.4 pandas==2.2.3 pyarrow==18.1.0 scikit-learn==1.5.2 psutil==6.1.0')

from pathlib import Path
import sys
PROJECT_ROOT = Path(PROJECT_ROOT)
DATASET_ROOT = Path(DATASET_ROOT)
DRIVE_ROOT = Path(DRIVE_ROOT)
sys.path.insert(0, str(PROJECT_ROOT / 'code_CICFLOW'))

from colab.colab_pipeline import (
    BRANCHES, CONFIGS, build_paths, check_source_layout,
    stage4_metadata_preflight, summarize_configs,
    verify_stage4_run, run_stage4_mi, run_stage5_rfe,
    backup_small_artifacts,
)
paths = build_paths(project_root=PROJECT_ROOT, dataset_root=DATASET_ROOT, drive_root=DRIVE_ROOT)
layout_status = check_source_layout(paths)
layout_status

## Metadata, schema, source identity, and resource gate

This cell is metadata-only. It checks the Train footer, schema, expected window count, resource availability, and the single-root rule. It does not read Train feature values.

In [ ]:
preflight = {}
if layout_status['status'] == 'PASS':
    for config in CONFIGS:
        preflight[config] = stage4_metadata_preflight(paths, config)
preflight

## Stage 4 MI — Train-only Top 100

In `dry_run`, `full`, and `resume` mode this section is intentionally skipped or guarded by the configuration above. For a full MI run, use new Run IDs if the listed Windows/Drive run already exists; never overwrite a completed run.

In [ ]:
mi_results = {}
if MODE in {'full', 'resume'}:
    if not CONFIRM_PROVISIONAL_RESEARCH:
        raise RuntimeError('MI full/resume requires provisional confirmation')
    for config in CONFIGS:
        mi_results[config] = run_stage4_mi(
            paths, config, RUN_IDS[config], mode=MODE,
            confirm_provisional_research=CONFIRM_PROVISIONAL_RESEARCH,
        )
elif MODE == 'verify':
    for config in CONFIGS:
        mi_results[config] = verify_stage4_run(paths, config, RUN_IDS[config])
else:
    mi_results = {'status': 'DRY_RUN_SKIPPED_MI_FIT'}
mi_results

## Stage 5A RF-RFE — Train-only full ranking 100 → 1

RF and SVM-RFE are separate branches. Each branch binds to its exact Stage 4 MI run, writes step-one checkpoints, and never opens Validation/Test. XGBoost is intentionally absent and remains BLOCKED.

In [ ]:
rfe_results = {}
if MODE in {'full', 'resume'}:
    if not CONFIRM_PROVISIONAL_RESEARCH:
        raise RuntimeError('RFE full/resume requires provisional confirmation')
    for config in CONFIGS:
        for branch in BRANCHES:
            rfe_results[f'{config}/{branch}'] = run_stage5_rfe(
                paths, config, RUN_IDS[config], branch, mode=MODE,
                git_commit=GIT_COMMIT,
                confirm_provisional_research=CONFIRM_PROVISIONAL_RESEARCH,
            )
elif MODE == 'verify':
    for config in CONFIGS:
        for branch in BRANCHES:
            rfe_results[f'{config}/{branch}'] = run_stage5_rfe(
                paths, config, RUN_IDS[config], branch, mode='verify',
            )
else:
    rfe_results = {'status': 'DRY_RUN_SKIPPED_RFE_FIT'}
rfe_results

## Drive backup and checkpoint integrity

Only small JSON/CSV/Markdown/marker artifacts are backed up. No Parquet dataset file is copied. Backup writes a completion marker last and refuses an existing backup target.

In [ ]:
backup_results = {}
if BACKUP_SMALL_ARTIFACTS and MODE in {'full', 'resume', 'verify'}:
    for config in CONFIGS:
        run_id = RUN_IDS[config]
        stage4_dir = paths['stage4_runs_root'] / config / run_id
        if (stage4_dir / 'MI_COMPLETE.json').exists():
            backup_results[f'{config}/stage4'] = backup_small_artifacts(
                stage4_dir, paths['backup_root'] / 'stage4' / config
            )
        for branch in BRANCHES:
            stage5_dir = paths['stage5_runs_root'] / config / run_id / branch
            if (stage5_dir / 'COMPLETE.json').exists():
                backup_results[f'{config}/stage5/{branch}'] = backup_small_artifacts(
                    stage5_dir, paths['backup_root'] / 'stage5' / config / branch
                )
backup_results

## Four-configuration summary

This summary is marker-based and does not promote a run to official reproduction.

In [ ]:
summary = summarize_configs(paths, RUN_IDS)
summary

## Stage 5B boundary

This notebook does not run Validation k-search, final Train+Validation refit, or Test evaluation. Follow `code_CICFLOW/stage5_rfe/reports/STAGE5B_CONTRACT.md` only after Stage 5A ranking and source compatibility are scientifically locked.